<a href="https://colab.research.google.com/github/JoshuaFZ/QWEN-0.6B-LORA/blob/main/voicesense_tunning_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SenseVoice 微调 Notebook（colab 版）
加载云盘安装环境

In [ ]:
from google.colab import drive
import os

# 1. 挂载 Google Drive
drive.mount('/content/drive')

# 2. 安装依赖
if 'MODE_WITH_AUTO_TEST' not in os.environ:
    !pip install -U modelscope -q
    !pip install addict soundfile librosa sentencepiece -q
    !pip install modelscope[audio] -f https://modelscope.oss-cn-beijing.aliyuncs.com/releases/repo.html -q

print("\n✅ 云盘已挂载，环境依赖安装完成！")


## 2. 在原有模型基础继续训练

In [ ]:
import os
import shutil
import subprocess
import sys
import torch
from pathlib import Path

print("--- 3. 重新执行微调 (增量训练) ---")
print("Notebook patch version: 2026-05-18-device-v3")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"当前计算设备: {device}")

# 1. 环境与路径准备
if not os.path.exists("./FunASR"):
    !git clone https://github.com/alibaba-damo-academy/FunASR.git

!pip install -U modelscope -q
!pip install addict soundfile librosa sentencepiece -q
!pip install -e ./FunASR -q

abs_cwd = os.getcwd()
abs_funasr_path = os.path.join(abs_cwd, "FunASR")

PROJECT_DIR = Path("/content/drive/MyDrive/SenseVoice_Project")
ORIGINAL_MODEL_PATH = PROJECT_DIR / "SenseVoiceSmall_Original"
PREV_FINETUNED_PATH = PROJECT_DIR / "sensevoice_finetuned_03"
DRIVE_OUTPUT_DIR = PROJECT_DIR / "sensevoice_finetuned_04"
TRAIN_JSONL = Path("/content/drive/MyDrive/branch1/train_with_len_plus_errors_20260519.jsonl")
BASE_MODEL_ID = "iic/SenseVoiceSmall"
local_training_dir = Path(abs_cwd) / "local_model_pkg"

def safe_cleanup(path):
    path = Path(path)
    if path.is_symlink() or path.is_file():
        path.unlink()
    elif path.is_dir():
        shutil.rmtree(path)

def copy_overlay(src, dst):
    src = Path(src)
    dst = Path(dst)
    if not src.exists():
        return
    for item in src.iterdir():
        target = dst / item.name
        safe_cleanup(target)
        if item.is_dir():
            shutil.copytree(item, target, symlinks=True)
        else:
            shutil.copy2(item, target)

def ensure_base_model():
    from modelscope import snapshot_download

    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    if ORIGINAL_MODEL_PATH.exists() and any(ORIGINAL_MODEL_PATH.iterdir()):
        print("已存在完整基座模型目录:", ORIGINAL_MODEL_PATH)
        return

    print("正在下载完整基座模型:", BASE_MODEL_ID)
    downloaded = Path(snapshot_download(BASE_MODEL_ID, cache_dir="/content/modelscope_cache"))
    safe_cleanup(ORIGINAL_MODEL_PATH)
    shutil.copytree(downloaded, ORIGINAL_MODEL_PATH, symlinks=True)
    print("基座模型已保存到 Google Drive:", ORIGINAL_MODEL_PATH)

if not TRAIN_JSONL.exists():
    raise FileNotFoundError(f"训练数据不存在: {TRAIN_JSONL}")

# 2. 下载/复用完整基座模型。不要用 AutoModel 预热缓存，避免触发 model 模块远程代码导入 warning。
ensure_base_model()

# 3. 构造完整本地模型包：先复制基座模型，再用上次微调输出覆盖。
safe_cleanup(local_training_dir)
shutil.copytree(ORIGINAL_MODEL_PATH, local_training_dir, symlinks=True)

if PREV_FINETUNED_PATH.exists() and any(PREV_FINETUNED_PATH.iterdir()):
    print("发现上次微调模型，覆盖到本地训练包:", PREV_FINETUNED_PATH)
    copy_overlay(PREV_FINETUNED_PATH, local_training_dir)
else:
    print("未发现上次微调模型，本次从完整基座模型开始训练。")

# 旧输出目录里的 config.yaml 可能带有 train_conf.device。
# FunASR train.py 会把顶层 device 单独传给 Trainer；train_conf.device 残留会导致 device 被传两次。
config_yaml_path = local_training_dir / "config.yaml"
if config_yaml_path.exists():
    try:
        import yaml

        config_data = yaml.safe_load(config_yaml_path.read_text(encoding="utf-8")) or {}
        train_conf = config_data.get("train_conf")
        if isinstance(train_conf, dict) and "device" in train_conf:
            removed_device = train_conf.pop("device")
            config_yaml_path.write_text(
                yaml.safe_dump(config_data, allow_unicode=True, sort_keys=False),
                encoding="utf-8",
            )
            print("已从 config.yaml 删除 train_conf.device:", removed_device)
    except Exception as exc:
        print("检查/清理 config.yaml 中 train_conf.device 失败:", exc)

# 让 trust_remote_code 能找到本地模型仓库里的 model.py 等远程代码文件。
if str(local_training_dir) not in sys.path:
    sys.path.insert(0, str(local_training_dir))

print("本地训练模型包:", local_training_dir)
print("包含文件:", sorted(p.name for p in local_training_dir.iterdir())[:30])

safe_cleanup(DRIVE_OUTPUT_DIR)

print(f"🚀 开始增量强化微调 (设备: {device})...")

env = os.environ.copy()
env["PYTHONPATH"] = f"{local_training_dir}:{abs_cwd}:{env.get('PYTHONPATH', '')}"
env["CUDA_VISIBLE_DEVICES"] = "0"
env["HYDRA_FULL_ERROR"] = "1"
env["PYTHONUNBUFFERED"] = "1"
TRAIN_LOG_PATH = PROJECT_DIR / "sensevoice_finetuned_resume_train.log"

train_cmd = [
    "torchrun",
    "--nproc_per_node=1",
    "FunASR/funasr/bin/train.py",
    "++model=" + str(local_training_dir),
    "++trust_remote_code=True",
    "++train_data_set_list=" + str(TRAIN_JSONL),
    "++valid_data_set_list=" + str(TRAIN_JSONL),
    "++dataset_conf.batch_type=token",
    "++dataset_conf.batch_size=2000",
    "++dataset_conf.data_split_num=1",
    "++dataset_conf.num_workers=0",
    "++dataset_conf.filter_conf.max_length=100000",
    "++dataset_conf.filter_conf.min_length=0",
    "++dataset_conf.filter_conf.token_max_length=100000",
    "++dataset_conf.filter_conf.token_min_length=0",
    "++train_conf.optim_conf.lr=0.00002",
    "++train_conf.max_epoch=5",
    "++device=" + str(device),
    "++output_dir=" + str(DRIVE_OUTPUT_DIR),
]

print("执行命令:")
print(" ".join(train_cmd))
if any("{local_training_dir}" in arg for arg in train_cmd):
    raise RuntimeError("训练命令仍包含未展开的 {local_training_dir}，请重新打开更新后的 notebook。")
if any("train_conf.device" in arg for arg in train_cmd):
    raise RuntimeError("训练命令仍包含 ++train_conf.device，请重新打开 2026-05-18-device-v3 版本 notebook。")

print("训练日志:", TRAIN_LOG_PATH)
with open(TRAIN_LOG_PATH, "w", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        train_cmd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
        log_file.write(line)
    return_code = process.wait()

if return_code != 0:
    print("\n❌ 训练失败，下面是日志最后 200 行:")
    try:
        tail_lines = TRAIN_LOG_PATH.read_text(encoding="utf-8", errors="replace").splitlines()[-200:]
        print("\n".join(tail_lines))
    except Exception as exc:
        print("读取训练日志失败:", exc)
    raise subprocess.CalledProcessError(return_code, train_cmd)

print(f"\n✅ 任务执行完毕！输出在 {DRIVE_OUTPUT_DIR}")


## 3.加载模型并测试

In [ ]:
import datetime
import html
import json
import os
import re
from pathlib import Path

import pandas as pd
from funasr import AutoModel
from IPython.display import HTML, display
from tqdm import tqdm


# --- 配置区 ---
DATA_ROOT = Path('/content/drive/MyDrive/branch1')
TEST_DIR = DATA_ROOT / 'asr_benchmark'
SOURCE_JSON_PATH = DATA_ROOT / 'wav_expected.json'
CSV_PATH = DATA_ROOT / 'error_analysis.csv'

PROJECT_DIR = Path('/content/drive/MyDrive/SenseVoice_Project')
RESUME_MODEL_DIR = PROJECT_DIR / 'sensevoice_finetuned_04'
PREV_FINETUNED_DIR = PROJECT_DIR / 'sensevoice_finetuned_02'

# 优先评估最新训练输出；如果还没有 resume 输出，则回退到之前的微调目录。
FINETUNED_MODEL_DIR = RESUME_MODEL_DIR if RESUME_MODEL_DIR.exists() else PREV_FINETUNED_DIR

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
HTML_PATH = Path(f'/content/asr_compare_with_paths_{timestamp}.html')


def clean_text(text):
    """清理 SenseVoice 控制符，保留用于展示的识别文本。"""
    if not text:
        return ''
    return re.sub(r'<\|.*?\|>', '', str(text)).replace(' ', '').strip()


def normalize_text(text):
    """忽略大小写、空格、标点和特殊符号后做自动判定。"""
    if not text:
        return ''
    text = str(text).lower()
    return re.sub(r'[^\w\u4e00-\u9fa5]', '', text)


def relative_audio_path(path):
    path = Path(path)
    try:
        return path.relative_to(DATA_ROOT).as_posix()
    except ValueError:
        return path.as_posix()


def load_expected_mapping(source_json_path):
    expected_mapping = {}
    basename_candidates = {}
    if not source_json_path.exists():
        print(f'⚠️ 未找到标注文件: {source_json_path}')
        return expected_mapping

    try:
        raw_data = json.loads(source_json_path.read_text(encoding='utf-8-sig'))
    except Exception as exc:
        print(f'⚠️ 读取 wav_expected.json 失败: {exc}')
        return expected_mapping

    if isinstance(raw_data, dict) and 'entries' in raw_data:
        data_list = raw_data['entries']
    elif isinstance(raw_data, list):
        data_list = raw_data
    elif isinstance(raw_data, dict):
        data_list = [{'file': key, 'expected': value} for key, value in raw_data.items()]
    else:
        data_list = []

    def normalize_key(value):
        return str(value).strip().replace('\\', '/').lstrip('./')

    def add_expected(key, text):
        key = normalize_key(key)
        if not key:
            return
        expected_mapping[key] = str(text).strip()
        if not key.startswith('asr_benchmark/'):
            expected_mapping[f'asr_benchmark/{key}'] = str(text).strip()
        basename_candidates.setdefault(Path(key).name, set()).add(str(text).strip())

    for item in data_list:
        audio_name = item.get('file') or item.get('wav') or item.get('audio')
        text = item.get('expected') or item.get('text') or item.get('txt') or item.get('label')
        if audio_name and text:
            add_expected(audio_name, text)

    # 只有文件名在标注中不冲突时，才允许用 basename 兜底。
    # clean/noise 等目录下存在同名 wav；直接按 basename 匹配会把 expected 覆盖错。
    for basename, values in basename_candidates.items():
        if len(values) == 1:
            expected_mapping[basename] = next(iter(values))

    return expected_mapping


def generate_text(model, wav_path):
    if model is None:
        return ''
    result = model.generate(input=str(wav_path), disable_pbar=True)
    if not result:
        return ''
    return clean_text(result[0].get('text', ''))


def evaluate_status(expected, base_text, finetuned_text):
    norm_expected = normalize_text(expected)
    if not norm_expected:
        return '无标准答案', '', ''

    base_correct = normalize_text(base_text) == norm_expected
    finetuned_correct = normalize_text(finetuned_text) == norm_expected

    if base_correct and finetuned_correct:
        status = '都对'
    elif base_correct:
        status = '基础对'
    elif finetuned_correct:
        status = '微调对'
    else:
        status = '都不对'

    return status, ' ✅' if base_correct else ' ❌', ' ✅' if finetuned_correct else ' ❌'


def status_html(status):
    if status in {'都对', '微调对'}:
        css_class = 'same'
    elif status in {'基础对', '都不对'}:
        css_class = 'diff'
    else:
        css_class = 'missing'
    return f"<span class='{css_class}'>{html.escape(status)}</span>"


def build_result_row(row, index):
    return f"""
        <tr>
            <td>{index}</td>
            <td><span class="path">{html.escape(row['rel_path'])}</span></td>
            <td><b>{html.escape(row['expected'])}</b></td>
            <td>{html.escape(row['base'])}{row['_base_mark']}</td>
            <td>{html.escape(row['ft'])}{row['_ft_mark']}</td>
            <td>{status_html(row['status'])}</td>
        </tr>
    """


print('--- 1. 加载模型 ---')
print('正在加载原始基础模型...')
base_model = AutoModel(model='iic/SenseVoiceSmall', trust_remote_code=True, disable_update=True)

print('\n正在加载微调后的模型...')
finetuned_model = None
if FINETUNED_MODEL_DIR.exists():
    try:
        finetuned_model = AutoModel(
            model=str(FINETUNED_MODEL_DIR),
            trust_remote_code=True,
            disable_update=True,
        )
        print('✅ 微调模型加载成功:', FINETUNED_MODEL_DIR)
    except Exception as exc:
        print(f'⚠️ 加载微调模型失败，可能是训练输出不完整: {exc}')
else:
    print('⚠️ 未找到微调模型目录:', FINETUNED_MODEL_DIR)

print(f'\n--- 2. 扫描目录 {TEST_DIR} ---')
wav_files = sorted(TEST_DIR.rglob('*.wav')) if TEST_DIR.exists() else []
print(f'共找到 {len(wav_files)} 个音频文件。开始批量测试...')

expected_mapping = load_expected_mapping(SOURCE_JSON_PATH)
print(f'已加载标注数量: {len(expected_mapping)}')

results = []
for wav_path in tqdm(wav_files, desc='推理进度'):
    try:
        file_name = wav_path.name
        rel_path = relative_audio_path(wav_path)
        try:
            test_rel_path = wav_path.relative_to(TEST_DIR).as_posix()
        except ValueError:
            test_rel_path = file_name
        expected = (
            expected_mapping.get(rel_path)
            or expected_mapping.get(test_rel_path)
            or expected_mapping.get(file_name, '')
        )
        base_text = generate_text(base_model, wav_path)
        finetuned_text = generate_text(finetuned_model, wav_path)
        status, base_mark, finetuned_mark = evaluate_status(expected, base_text, finetuned_text)

        results.append({
            'rel_path': rel_path,
            'file': file_name,
            'expected': expected,
            'base': base_text,
            'ft': finetuned_text,
            'status': status,
            'path': str(wav_path),
            'corrected_text': '',
            '_base_mark': base_mark,
            '_ft_mark': finetuned_mark,
        })
    except Exception as exc:
        print(f'文件 {wav_path} 推理出错: {exc}')

print('\n--- 3. 导出 CSV 和 HTML ---')
df = pd.DataFrame(results)
export_columns = ['rel_path', 'file', 'expected', 'base', 'ft', 'status', 'path', 'corrected_text']
if not df.empty:
    df[export_columns].to_csv(CSV_PATH, index=False, encoding='utf-8-sig')
else:
    pd.DataFrame(columns=export_columns).to_csv(CSV_PATH, index=False, encoding='utf-8-sig')

rows_html = []
for i, row in enumerate(results, 1):
    rows_html.append(build_result_row(row, i))

status_counts = df['status'].value_counts().to_dict() if not df.empty else {}
summary_rows_html = []
for status in ['都对', '微调对', '基础对', '都不对', '无标准答案']:
    summary_rows_html.append(f"""
        <tr>
            <td>{status_html(status)}</td>
            <td>{status_counts.get(status, 0)}</td>
        </tr>
    """)

finetune_failures = [
    row for row in results
    if row['status'] in {'基础对', '都不对'} or not row['ft']
]
failure_rows_html = []
for i, row in enumerate(finetune_failures, 1):
    failure_rows_html.append(build_result_row(row, i))

failure_section_html = ''.join(failure_rows_html) if failure_rows_html else """
        <tr>
            <td colspan="6" class="empty">没有需要重点查看的微调失败样本。</td>
        </tr>
"""

html_content = f"""<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>ASR 对比测试报告</title>
    <style>
        body {{ font-family: Arial, sans-serif; margin: 20px; }}
        table {{ border-collapse: collapse; width: 100%; margin-top: 20px; }}
        th, td {{ border: 1px solid #ddd; padding: 10px; text-align: left; vertical-align: top; }}
        th {{ background-color: #f2f2f2; }}
        .diff {{ color: #c00000; font-weight: bold; }}
        .same {{ color: #087a1f; font-weight: bold; }}
        .missing {{ color: #666; }}
        .path {{ font-size: 0.85em; color: #666; word-break: break-all; }}
        .empty {{ color: #666; text-align: center; }}
        .section-title {{ margin-top: 28px; }}
    </style>
</head>
<body>
    <h2>SenseVoice 微调前后对比测试报告</h2>
    <p><b>测试目录:</b> {html.escape(str(TEST_DIR))}</p>
    <p><b>微调模型:</b> {html.escape(str(FINETUNED_MODEL_DIR))}</p>
    <p><b>测试文件数:</b> {len(results)}</p>
    <p><b>微调失败/需重点查看:</b> {len(finetune_failures)}</p>
    <p><b>生成时间:</b> {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
    <p><i>状态判定会忽略大小写、空格、标点和特殊符号差异，并与 wav_expected.json 中的 expected_text 比较。</i></p>

    <h3 class="section-title">状态汇总</h3>
    <table>
        <tr>
            <th>状态</th>
            <th>数量</th>
        </tr>
        {''.join(summary_rows_html)}
    </table>

    <h3 class="section-title">微调失败/需重点查看样本</h3>
    <p><i>这里集中列出“基础对但微调错”“基础和微调都错”以及微调输出为空的样本。</i></p>
    <table>
        <tr>
            <th>序号</th>
            <th>相对路径</th>
            <th>原始标注 Expected</th>
            <th>原始基础模型</th>
            <th>微调后模型</th>
            <th>判定结果</th>
        </tr>
        {failure_section_html}
    </table>

    <h3 class="section-title">完整明细</h3>
    <table>
        <tr>
            <th>序号</th>
            <th>相对路径</th>
            <th>原始标注 Expected</th>
            <th>原始基础模型</th>
            <th>微调后模型</th>
            <th>判定结果</th>
        </tr>
        {''.join(rows_html)}
    </table>
</body>
</html>
"""

HTML_PATH.write_text(html_content, encoding='utf-8')

print(f'✅ HTML 报告已生成: {HTML_PATH}')
print(f'✅ CSV 文件已导出至: {CSV_PATH}')
print('\n💡 人工清洗与微调指南')
print("1. 打开 error_analysis.csv。")
print("2. 重点关注 status 为 '都不对' 或 '基础对' 的样本。")
print("3. 在 corrected_text 列填写完全正确的文本；HTML 报告用于浏览检查，CSV 用于后续脚本处理。")

display(HTML(
    f"<b style='color:green;'>对比报告生成成功。</b><br>"
    f"HTML: <code>{html.escape(str(HTML_PATH))}</code><br>"
    f"CSV: <code>{html.escape(str(CSV_PATH))}</code>"
))
